# Slice docs the way Google wishes you would

This notebook compares fixed-size, recursive, and token-based chunks using a real short podcast episode and an open-access ILO PDF. The podcast audio is in Polish; transcription keeps the original language and timestamps. The publisher's English page is cited for source context, but retrieval uses the transcript generated from the audio. No API key is needed.

In [ ]:
%pip install -q langchain-text-splitters pypdf tiktoken matplotlib pandas requests faster-whisper

In [ ]:
from pathlib import Path
import requests

DATA=Path("data"); DATA.mkdir(exist_ok=True)
AUDIO_URL="https://kamil.org.pl/wp-content/uploads/2026/10/kamil-podcast-odcinek-01.mp3"
PODCAST_PAGE="https://kamil.org.pl/en/2026/10/12/podcast-episode-01-hello/"
PDF_URL="https://www.ilo.org/sites/default/files/2024-07/WP96_web.pdf"
AUDIO=DATA/"podcast.mp3"; TRANSCRIPT=DATA/"podcast_transcript.txt"; PDF=DATA/"ilo_working_paper_96.pdf"
def download(url,path):
    if not path.exists():
        with requests.get(url,stream=True,timeout=60) as response:
            response.raise_for_status()
            with path.open("wb") as f:
                for block in response.iter_content(1024*1024):
                    if block: f.write(block)
    print(path, path.stat().st_size, "bytes")
download(AUDIO_URL,AUDIO); download(PDF_URL,PDF)

In [ ]:
# Local Whisper transcription; transcript includes segment timestamps and is saved locally.
# This PyAV compatibility wrapper accommodates versions that don't accept metadata_errors.
import av
_av_open=av.open
def _av_open_compat(*args,metadata_errors=None,**kwargs):
    return _av_open(*args,**kwargs)
av.open=_av_open_compat
from faster_whisper import WhisperModel
model=WhisperModel("base",device="cpu",compute_type="int8")
segments,info=model.transcribe(str(AUDIO),vad_filter=True)
transcript_lines=[f"[{s.start:05.1f}-{s.end:05.1f}s] {s.text.strip()}" for s in segments]
transcript="\n".join(transcript_lines)
TRANSCRIPT.write_text(transcript+"\n",encoding="utf-8")
print(f"Detected language: {info.language} ({info.language_probability:.3f}); duration: {info.duration:.1f}s")
print(transcript)

In [ ]:
from pypdf import PdfReader
from langchain_core.documents import Document
reader=PdfReader(str(PDF))
pdf_page_docs=[Document(page_content=(page.extract_text() or ""),metadata={"source":"ILO Working Paper 96","page":i+1})
               for i,page in enumerate(reader.pages)]
pdf_page_docs=[d for d in pdf_page_docs if d.page_content.strip()]
podcast_doc=Document(page_content=transcript,metadata={"source":"Kamil podcast episode 01","language":info.language,"audio_url":AUDIO_URL,"page_url":PODCAST_PAGE})
print("Podcast transcript characters:",len(transcript))
print("PDF pages extracted:",len(pdf_page_docs),"characters:",sum(len(d.page_content) for d in pdf_page_docs))
print("PDF title:",reader.metadata.title)

## Chunk strategies and comparison

Character-based sizes are measured in characters. Token-based sizes use the `cl100k_base` tokenizer. PDF page metadata and podcast language/source metadata are retained on each chunk.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter, TokenTextSplitter
import pandas as pd

def make_chunks(docs,strategy,size,overlap):
    if strategy=="fixed":
        splitter=CharacterTextSplitter(separator=" ",chunk_size=size,chunk_overlap=overlap)
    elif strategy=="recursive":
        splitter=RecursiveCharacterTextSplitter(chunk_size=size,chunk_overlap=overlap,
            separators=["\n\n","\n",". ","? ","! "," ",""])
    else:
        splitter=TokenTextSplitter(encoding_name="cl100k_base",chunk_size=size,chunk_overlap=overlap)
    return splitter.split_documents(docs)

def pct_clean_boundaries(chunks):
    nonempty=[c for c in chunks if c.page_content.strip()]
    return 100*sum(c.page_content.strip()[-1] in ".?!" for c in nonempty)/len(nonempty) if nonempty else 0

def stats(chunks):
    sizes=[len(c.page_content) for c in chunks]
    return {"chunks":len(chunks),"avg_chars":round(sum(sizes)/len(sizes),1) if sizes else 0,
            "min_chars":min(sizes,default=0),"max_chars":max(sizes,default=0),
            "clean_endings_pct":round(pct_clean_boundaries(chunks),1)}

source_docs={"Podcast transcript":[podcast_doc],"ILO PDF":pdf_page_docs}
chunk_sets={}; rows=[]
for source,docs in source_docs.items():
    chunk_sets[source]={}
    for size in (500,1000,2000):
        for overlap in (0,50,100):
            name=f"Fixed-{size}-overlap-{overlap}"
            chunks=make_chunks(docs,"fixed",size,overlap); chunk_sets[source][name]=chunks
            rows.append({"source":source,"strategy":name,**stats(chunks)})
        name=f"Recursive-{size}-overlap-50"
        chunks=make_chunks(docs,"recursive",size,50); chunk_sets[source][name]=chunks
        rows.append({"source":source,"strategy":name,**stats(chunks)})
    for size in (500,1000):
        name=f"Token-{size}-overlap-50"
        chunks=make_chunks(docs,"token",size,50); chunk_sets[source][name]=chunks
        rows.append({"source":source,"strategy":name,**stats(chunks)})
comparison=pd.DataFrame(rows)
comparison

In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,source in zip(axes,["Podcast transcript","ILO PDF"]):
    sample=chunk_sets[source]["Recursive-1000-overlap-50"]
    ax.hist([len(c.page_content) for c in sample],bins=20)
    ax.set_xlabel("Chunk size (characters)"); ax.set_ylabel("Frequency"); ax.set_title(source+" — Recursive 1000")
plt.tight_layout(); plt.show()

In [ ]:
for source in chunk_sets:
    sample=chunk_sets[source]["Recursive-500-overlap-50"]
    print("\n",source,"— example chunk tails")
    for i,chunk in enumerate(sample[:3]):
        print(i,chunk.metadata,repr(chunk.page_content[-50:]))

## One retrieval trace

The selected path is recursive 500-character chunking with 50-character overlap. This transparent word-overlap retriever is a small inspection aid, not production semantic search.

In [ ]:
query="O czym będzie podcast?"  # What will the podcast be about?
def words(text):
    import re
    return {w.casefold() for w in re.findall(r"[\w]+",text,flags=re.UNICODE) if len(w)>2}
qwords=words(query)
selected=chunk_sets["Podcast transcript"]["Recursive-500-overlap-50"]
hits=[]
for i,doc in enumerate(selected):
    score=len(qwords & words(doc.page_content))
    if score: hits.append((score,i,doc))
hits.sort(key=lambda x:(-x[0],x[1]))
for score,i,doc in hits[:2]:
    print(f"[podcast#chunk-{i} | overlap={score} | source={doc.metadata['source']} | language={doc.metadata['language']}]\n{doc.page_content}\n")

In [ ]:
final_answer=("Kamil says the podcast will cover local AI models, automation of everyday work, tools he tests, and his conclusions. "
              "This answer is based on the transcribed audio segment shown above; the transcript is Polish.")
print(final_answer)
print("Limitation: word overlap can rank a chunk that shares terms without supporting the answer. Check the timestamped source passage before accepting it.")

## Recommendations

**PDF:** Start with recursive chunks around 1,000 characters and 50–100 overlap. Preserve page numbers; recursive boundaries tend to retain headings and paragraphs, but inspect pages with tables or multi-column layout.

**Podcast transcript:** Start with recursive chunks around 500–1,000 characters and 50 overlap. Keep timestamps and language metadata so a reviewer can return to the audio; check speaker turns and ASR errors.

| Strategy | Pros | Cons | Best fit |
|---|---|---|---|
| Fixed size | Simple and predictable | Can cut sentences, turns, or headings | Baseline and uniform text |
| Recursive | Tries paragraph/line/sentence boundaries first | Depends on source formatting and separator order | Structured PDFs and transcripts |
| Token based | Controls context-window size | Tokenizer-specific; can split awkwardly | LLM context budgeting |
| Semantic | Can group by meaning | Requires embeddings and threshold tuning | Long or topic-shifting material |

Clean sentence endings are only a boundary heuristic. Retrieval relevance and direct source inspection are needed to assess grounding.